# bge-m3 건강 상담 문서 임베딩 (Colab GPU)

`scripts/compare_kure.py --bge-m3`가 쓸 `BAAI/bge-m3`(공개 모델, 1024차원) 문서 벡터를 만든다. 건강 컬렉션을 bge-m3로 바꿀 때의 효과는 재지 않았다(`docs/wiki/open-questions.md`). 같은 계열인 KURE-v1(bge-m3를 한국어 검색으로 더 학습)은 이미 쟀다.

- 문서는 1,024토큰까지 넣는다(KURE-v1과 같다). 상담 글 19,206건 중 약 0.1%만 잘린다.
- 접두어는 넣지 않는다(bge-m3 dense 사용법).
- 런타임: GPU T4 이상. 모델 약 2.3GB를 받는다. float16으로 돌린다.
- 예상: GPU 메모리 최대 약 3~6GB(T4 15GB의 절반 이하), 문서 임베딩 약 5~10분, 패키지 설치와 모델 받기를 더해 모두 15분 안쪽.
- HF 토큰: Colab Secrets의 `HF_TOKEN`(없으면 실행할 때 입력). 비공개 데이터셋에서 `df.csv`를 받고 결과를 올리는 데 쓴다.
- 결과: 비공개 데이터셋 `blanden77/dograg-data`의 `experiments` 브랜치에 `bgem3/bge_m3_rows.npy`(float32, 정규화)로 올린다. 행 순서 = `df.csv` 행 번호 = 앱 문서 ID.
- 다음(로컬): `uv run --group tools python scripts/compare_kure.py --bge-m3 --dry-run`으로 판사 호출 수를 먼저 본다.


In [ ]:
!pip -q install -U sentence-transformers huggingface_hub
import torch; print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU 없음: 런타임 유형을 GPU로 바꾸세요')


In [ ]:
import getpass
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None
HF_TOKEN = (HF_TOKEN or getpass.getpass('HF_TOKEN (영문 입력 상태에서 붙여 넣기): ')).strip()
# A token typed with the Korean input mode on turns into Hangul and fails later as UnicodeEncodeError.
if not HF_TOKEN.isascii() or not HF_TOKEN.startswith('hf_'):
    raise ValueError(f'HF_TOKEN이 hf_로 시작하는 영문·숫자 값이 아닙니다(길이 {len(HF_TOKEN)}, 한글 섞임: {not HF_TOKEN.isascii()}). '
                     'Hugging Face 토큰을 복사해 Colab Secrets의 HF_TOKEN에 붙여 넣으세요.')
print('HF_TOKEN ok, length', len(HF_TOKEN))


In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download
REPO, REVISION = 'blanden77/dograg-data', 'data-v12'  # src/private_data.py
path = hf_hub_download(REPO, 'data/df.csv', repo_type='dataset', revision=REVISION, token=HF_TOKEN)
df = pd.read_csv(path)
# The app's documents are qa.input with carriage returns removed (19,206 of 19,206 match the Chroma texts).
CR = chr(13)
texts = df['qa.input'].astype(str).str.replace(CR, '', regex=False).tolist()
assert len(texts) == 19206, len(texts)
print(len(texts))


In [ ]:
import time, numpy as np
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('BAAI/bge-m3', device='cuda').half()  # float16 on the GPU: about twice as fast
model.max_seq_length = 1024
cut = sum(len(ids) > 1024 for ids in model.tokenizer(texts, add_special_tokens=True)['input_ids'])
print(f'posts over 1,024 tokens: {cut} ({cut / len(texts):.2%})')
# Shortest first: batches of similar length waste less padding.
order = np.argsort([len(t) for t in texts])
torch.cuda.reset_peak_memory_stats()
started = time.time()
sorted_vectors = model.encode([texts[i] for i in order], batch_size=64, normalize_embeddings=True,
                              show_progress_bar=True, convert_to_numpy=True).astype(np.float32)
vectors = np.empty_like(sorted_vectors)
vectors[order] = sorted_vectors
print(vectors.shape, f'{time.time() - started:.0f}s', f'GPU peak {torch.cuda.max_memory_allocated() / 2**30:.1f} GB',
      'norm', float(np.linalg.norm(vectors[:3], axis=1).mean()))
np.save('bge_m3_rows.npy', vectors)


In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
api.create_branch(REPO, repo_type='dataset', branch='experiments', exist_ok=True)
api.upload_file(path_or_fileobj='bge_m3_rows.npy', path_in_repo='bgem3/bge_m3_rows.npy',
                repo_id=REPO, repo_type='dataset', revision='experiments')
print('uploaded to', REPO, '@ experiments')
